# Building the evaluation set: v2, then v3

The measuring stick, rebuilt twice.

**Read the note at the seam before treating this as one sitting.** The two halves
did not happen back to back -- the embedding re-evaluation in
`18. embeddings_evaluation_v2.ipynb` runs *between* them, and it is the reason
v3 exists at all.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
KB = PROJECT / "0. cleaned_data" / "Knowledgebase"

# source file -> (law code used in chunks_articles.parquet, language)
SOURCES = {
    "scl_articles_ar.xlsx":          ("ssl", "ar"),
    "scl_articles_en.xlsx":          ("ssl", "en"),
    "worklaw_ar_articles.xlsx":      ("labor96", "ar"),
    "worklaw_articles_en.xlsx":      ("labor96", "en"),
    "worklaw_prof_ar_articles.xlsx": ("prof19", "ar"),
    "worklaw_prof_articles_en.xlsx": ("prof19", "en"),
}

kb = {}
for fn, (law, lang) in SOURCES.items():
    df = pd.read_excel(KB / fn)
    df["Article #"] = df["Article #"].astype(str).str.strip()   # dtype is str in SCL, int64 in labor -- normalise
    kb[(law, lang)] = df.set_index("Article #")
    print(f"{fn:32s} -> {law:8s} {lang}  {len(df):4d} articles")

scl_articles_ar.xlsx             -> ssl      ar   111 articles
scl_articles_en.xlsx             -> ssl      en   111 articles
worklaw_ar_articles.xlsx         -> labor96  ar   142 articles
worklaw_articles_en.xlsx         -> labor96  en   142 articles
worklaw_prof_ar_articles.xlsx    -> prof19   ar    12 articles
worklaw_prof_articles_en.xlsx    -> prof19   en    13 articles


## The 60 questions

Each entry is `(question_text, law, [article_numbers], category, difficulty, labeling_note)`. Language is implied by the block. `labeling_note` is the ground-truth answer written from the source text — it is the anchor everything else gets scored against, so it states the actual numbers and conditions rather than paraphrasing loosely.

In [2]:
# --- Social Security Law, ENGLISH, 15 questions, chapters 4-8 (articles 23-77 by content) ---
SCL_EN = [
    ("What benefits does Work Injury Insurance provide to an injured worker?",
     "ssl", ["25"], "benefits", "easy",
     "Work Injury Insurance covers four things: medical care required by the injured person's health condition; a daily allowance for temporary inability to work; monthly pensions and lump sum compensation for the injured and entitled beneficiaries; and funeral expenses if the injury causes death."),

    ("Does the medical care provided under Work Injury Insurance include transport costs and prosthetic devices?",
     "ssl", ["26"], "benefits", "medium",
     "Yes to both. Medical care under Article 26 includes treatment and hospitalisation costs, transport expenses to and from the workplace or residence to the place of treatment, and rehabilitation services and equipment including prosthetic devices, with the type and degree determined by the Medical Reference."),

    ("What obligations must the employer fulfil immediately following a work injury?",
     "ssl", ["27"], "obligations", "medium",
     "The firm must transport the injured insured to a treatment facility approved by the Corporation, report reportable injuries to the nearest police station within 48 hours, and notify the Corporation in writing with the initial medical report within 14 working days. If it fails to report to the Corporation on time it must cover 15% of the injured person's treatment expenses."),

    ("What daily allowance is payable to an insured person who is unable to work as a result of a work injury?",
     "ssl", ["29"], "benefits_calculation", "easy",
     "A daily allowance equal to 75% of the daily wage on which contributions were calculated at the date of injury, paid for the days spent under treatment at an approved centre or at home per a Medical Reference decision. It continues while the injured is unable to resume work, or until permanent disability is confirmed or death occurs."),

    ("What pension or compensation is payable for death, total disability, or partial disability resulting from a work injury?",
     "ssl", ["30"], "benefits_calculation", "hard",
     "Article 30 distinguishes four cases. (a) Death due to work injury: survivor beneficiaries receive a pension of 75% of the wage used to calculate contributions at the date of injury, accruing from the start of the month in which death occurs. (b) Total permanent work-related disability: a monthly pension of 75% of that same wage, accruing from the date the Medical Reference confirms the condition has stabilised, increased by 25% if the Medical Reference decides the injured needs assistance to perform daily life tasks -- that increase capped at the approved minimum wage under the Labor Law. (c) Partial permanent disability of 30% or more: a monthly pension equal to the disability percentage applied to the paragraph (b) total-disability pension, from the date of health stability. (d) Partial permanent disability below 30%: a lump sum equal to that proportion of the paragraph (b) amount, multiplied by 36 months."),

    ("In what cases does entitlement to work injury compensation lapse because of the conduct of the injured person?",
     "ssl", ["31"], "eligibility", "medium",
     "Entitlement to the daily allowance and the lump sum compensation lapses if the injury resulted from a deliberate act; from being under the influence of alcohol, narcotics, psychotropic substances or dangerous drugs; or from failing to comply with announced mandatory treatment or occupational safety instructions where that non-compliance was the main cause of the injury, a significant factor in it, or delayed recovery. These must be confirmed by an investigation carried out or approved by the Corporation."),

    ("May an injured worker claim compensation from the employer beyond what the Social Security Law provides?",
     "ssl", ["37"], "rights", "medium",
     "No. Neither the injured nor their heirs or survivor beneficiaries may recourse to the firm for work-injury compensation beyond what this Law provides -- unless the injury resulted from a serious mistake on the part of the firm."),

    ("Does coverage continue if symptoms of an occupational disease appear after the insured has left that occupation?",
     "ssl", ["39"], "eligibility", "medium",
     "Yes. The Corporation remains bound by the insurance rights in this Law if symptoms of an occupational disease appear within 2 years from the date work at that particular occupation ended, provided a Medical Reference decision confirms the occupation or job caused the disease."),

    ("May an insured person combine earnings from employment with a work injury disability pension?",
     "ssl", ["40"], "rights", "easy",
     "Yes. The insured may combine earnings from a job or occupation with their Work Injury Disability Pension allocated under this Law."),

    ("What are the conditions for entitlement to the maternity allowance, and how is its amount determined?",
     "ssl", ["44", "45"], "benefits_calculation", "hard",
     "Two articles combine. Article 44: the female insured must have been subject to Maternity Insurance for the last 6 months before entitlement, or have 6 continuous or discontinuous contributions in the 12 months before it, and childbirth must be established by official certificate. Article 45: the allowance during maternity leave equals her last wage subject to deduction at the start of the leave, and the leave counts as an actual service period, with Old-age/Disability/Death and Unemployment contributions deducted from the allowance."),

    ("What are the conditions for entitlement to the unemployment allowance?",
     "ssl", ["50"], "eligibility", "easy",
     "The insured must have at least 36 contributions before the date of entitlement, including at least 1 contribution toward Unemployment Insurance, and must not yet have reached age 60 for a male or 55 for a female."),

    ("For what period is the unemployment allowance payable, and at what percentages of the wage?",
     "ssl", ["52", "53"], "benefits_calculation", "hard",
     "Two articles combine. Article 52: 3 months if the insured has fewer than 180 contributions, 6 months if 180 or more; the period counts as actual service. Article 53: the allowance is calculated from the last wage subject to deduction at 75% for the first month, 65% for the second, 55% for the third, and 45% for the fourth, fifth and sixth -- capped at a net JOD 500 per month, adjusted annually to inflation."),

    ("May the unemployment allowance be combined with a maternity allowance, a lump sum compensation, or a retirement pension?",
     "ssl", ["56"], "rights", "medium",
     "No, in three separate cases. A female insured may not combine unemployment allowance with maternity leave allowance -- the higher one is paid. The insured may not combine it with the lump sum compensation under Article 70. And the insured may not combine it with any Retirement Pension or Disability Pension allocated under this Law."),

    ("What are the age and contribution requirements for entitlement to the Old-Age Pension?",
     "ssl", ["62"], "eligibility", "medium",
     "Old-Age Pension requires age 60 for a male or 55 for a female, and at least 180 paid contributions for Old-age Insurance of which 84 must be for actual periods of service. The pension is calculated per contribution year at 2.5% of the first JOD 1500 of the average monthly wage over the last 36 contributions, and 2% of the portion above JOD 1500."),

    ("What are the routes to an Early Retirement Pension, and how is it calculated?",
     "ssl", ["64"], "eligibility", "hard",
     "Article 64 provides several routes, all on the insured's request after service ends for any reason. (a) Age 50 or above with at least 252 actual contributions for a male or 228 for a female; or 300 actual contributions at age 45. (b) Transitional cases for those who had reached a given age before the Law took effect -- for example age 46 with at least 228 actual contributions (male) or 192 (female) for someone who had turned 44 before that date, with equivalent steps at ages 47 and 48. (c) Hazardous professions: age 45 or above with at least 216 actual contributions (male) or 180 (female), where the insured worked in that profession for at least 60 contributions in the preceding 10 years; the firm pays an extra 1% of wage. (d) Calculation: 2.5% per contribution year on the first JOD 1500 of the average monthly wage over the last 60 contributions and 2% on the remainder, then reduced by age according to appended Table 5, with dependant increases of 12% for the first dependant (JOD 10-100) and 6% each for the second and third (JOD 10-25). (h) For anyone first covered after the amended Law took effect: male age 55 with at least 252 actual contributions, female age 52 with at least 228."),
]
print(len(SCL_EN))

15


In [3]:
# --- Social Security Law, ARABIC, 15 questions, chapters 4-8 ---
# Formal Modern Standard Arabic, matching the register of the English block and of the corpus
# itself. Deliberately different articles from the English block: if both languages asked the
# same 15 things, this would measure translation consistency rather than independent retrieval.
SCL_AR = [
    ("من يتحمل اشتراك تأمين إصابات العمل، العامل أم صاحب العمل؟",
     "ssl", ["24"], "scope", "easy",
     "المنشأة وحدها تدفع اشتراكاً شهرياً بنسبة 2% من أجور المؤمن عليهم، ولا يُقتطع شيء من أجر العامل لهذا التأمين. وتشمل مصادر التمويل أيضاً الفوائد والغرامات على عدم التقيد بأحكام هذا التأمين، وريع استثمار الأموال المتأتية من هذه المصادر."),

    ("إلى متى تبقى المؤسسة مسؤولة عن علاج المصاب بعد إصابة العمل؟",
     "ssl", ["28"], "benefits", "medium",
     "تتولى المؤسسة علاج المصاب إلى أن يثبت شفاؤه أو عجزه أو وفاته الناشئة عن الإصابة بقرار من المرجع الطبي. وإذا استقرت الحالة بثبوت عجز كلي أو جزئي إصابي دائم، يقرر المرجع الطبي الأدوية والمستلزمات الطبية اللازمة لاستمرار حياته وتتحمل المؤسسة نفقاتها. وإذا انتكس المصاب خلال سنة من استقرار حالته يقرر المرجع الطبي حاجته للعلاج مجدداً واستحقاقه للبدل اليومي."),

    ("إذا تكررت إصابة العمل بعد إصابة سابقة، كيف يُحتسب التعويض أو راتب الاعتلال؟",
     "ssl", ["35"], "benefits_calculation", "hard",
     "تتوقف القاعدة على نسبة العجز الإجمالي الناشئ عن الإصابة الحالية والإصابات السابقة مجتمعة. (أ) إذا كانت النسبة أقل من 30% يُدفع للمصاب تعويض عن نسبة العجز الناشئ عن الإصابة الأخيرة وحدها، ويُحسب على أساس الأجر المعتمد لتسديد الاشتراكات بتاريخ وقوع الإصابة الأخيرة ووفقاً لأحكام الفقرة (د) من المادة 30. (ب) إذا بلغت النسبة 30% فأكثر فيُحسب راتب اعتلال على أساس نسبة العجز الناشئ عن الإصابات جميعاً وفقاً للأجر المعتمد بتاريخ وقوع الإصابة الأخيرة، وذلك في حالتين: إذا كان المصاب قد سبق له الحصول على تعويض عن إصابة أو إصابات سابقة؛ أو إذا كان يتقاضى راتب اعتلال عن إصابة أو إصابات سابقة، وفي هذه الحالة الثانية يُشترط ألا يقل راتب الاعتلال الجديد عما كان يتقاضاه قبل وقوع الإصابة الأخيرة."),

    ("هل يجوز إعادة الفحص الطبي بعد تقرير راتب الاعتلال، وما المدة المحددة لذلك؟",
     "ssl", ["36"], "procedure", "medium",
     "نعم. يجوز لكل من المؤسسة والمصاب الذي تقرر له راتب اعتلال عجز جزئي أو كلي إصابي دائم طلب إعادة الفحص الطبي مرة واحدة على الأقل كل ستة أشهر خلال السنتين التاليتين لتاريخ ثبوت العجز. وإذا أصبحت نسبة العجز أقل من 30% يوقف صرف راتب الاعتلال نهائياً ويُمنح المصاب التعويض المقرر عن نسبة العجز التي لم يسبق تعويضه عنها."),

    ("هل يجوز الاعتراض على قرار اللجنة الطبية الأولية بشأن إصابة العمل، وما المدة المحددة لتقديم الاعتراض؟",
     "ssl", ["38"], "procedure", "medium",
     "نعم، لكل من المؤسسة والمؤمن عليه الحق في الاعتراض على قرارات اللجنة الطبية الأولية أمام اللجنة الطبية الاستئنافية. ويُقدَّم الاعتراض خلال ستين يوماً من اليوم التالي لتاريخ التبليغ بالقرار، مرفقاً بالوثائق اللازمة، ولا تُقبل أي وثائق تُقدَّم بعد هذا التاريخ."),

    ("إذا وقعت إصابة العمل بفعل الغير، هل تبقى المؤسسة مسؤولة تجاه المصاب؟",
     "ssl", ["41"], "scope", "medium",
     "نعم، تبقى مسؤولية المؤسسة قائمة تجاه المصاب المؤمن عليه، وللمؤسسة الرجوع على الغير للمطالبة بكامل ما دفعته من تكاليف العناية الطبية والبدلات اليومية، وفقاً للأنظمة الصادرة بمقتضى أحكام هذا القانون."),

    ("هل يوقف صرف بدل إجازة الأمومة إذا التحقت المؤمن عليها بعمل خلال الإجازة؟",
     "ssl", ["46"], "rights", "easy",
     "نعم، يوقف صرف بدل إجازة الأمومة في حال التحاق المؤمن عليها بعمل خلال هذه الإجازة. لكن يجوز لها الجمع بين بدل إجازة الأمومة وراتب اعتلال العجز الجزئي أو الكلي الإصابي الدائم المستحق لها."),

    ("ما الحكم إذا عاد المؤمن عليه إلى العمل أثناء تقاضيه بدل التعطل عن العمل؟",
     "ssl", ["51"], "obligations", "medium",
     "يوقف صرف بدل التعطل عن العمل إذا ثبت للمؤسسة أن المؤمن عليه عاد إلى عمل يتقاضى عنه أجراً أو زاول عملاً تجارياً أو مهنياً لحسابه الخاص. وإذا ثبت أنه تقاضى البدل دون وجه حق تسترد المؤسسة جميع المبالغ المصروفة له مع الغرامة التي تحددها الأنظمة، محسوبة من تاريخ الصرف حتى تاريخ الاسترداد."),

    ("متى يبدأ صرف بدل التعطل عن العمل؟",
     "ssl", ["54"], "procedure", "easy",
     "يبدأ الصرف اعتباراً من أول الشهر التالي للشهر الذي تقدّم فيه المؤمن عليه بطلب صرف البدل. ولا يُصرف بدل التعطل عن الشهر الذي أُنهيت فيه خدمة المتعطل عن العمل."),

    ("كم مرة يستحق المؤمن عليه بدل التعطل عن العمل طيلة مدة شموله بأحكام القانون؟",
     "ssl", ["55"], "eligibility", "medium",
     "ثلاث مرات طيلة مدة شمول المؤمن عليه بأحكام القانون. ويُصرف البدل في أي من المرتين اللاحقتين للمرة الأولى إذا كان عدد اشتراكاته في كل منهما لا يقل عن 36 اشتراكاً لاحقاً لصرف البدل السابق."),

    ("ما مصير المبالغ المقتطعة لتأمين التعطل عن العمل عند الخروج النهائي من أحكام القانون؟",
     "ssl", ["57"], "benefits", "hard",
     "تُعتبر إيرادات تأمين التعطل عن العمل حساباً ادخارياً للمؤمن عليه وتتم تسوية حقوقه من هذا الحساب عند خروجه نهائياً من أحكام القانون. يُعاد إليه الرصيد المتراكم المتكوّن من الاشتراكات المقتطعة من أجره والاشتراكات المدفوعة من المنشأة مضافاً إليها الريع الاستثماري المتحقق على المبالغ المستثمرة، مخصوماً منها جميع المبالغ المصروفة له مثل بدل التعطل ونفقات إدارة الحساب الادخاري، وفق الأسس التي يحددها المجلس. وإذا كان الحساب الادخاري مديناً يُعاد إليه ما يقابل ذلك وفق ما تنص عليه المادة."),

    ("إذا بلغ المؤمن عليه سن الستين ولم يكمل الاشتراكات اللازمة لراتب التقاعد، هل يجوز له الاستمرار في الشمول بأحكام القانون؟",
     "ssl", ["63"], "eligibility", "medium",
     "نعم. للمؤمن عليه الاستمرار في شموله بأحكام القانون أو الالتحاق بعمل جديد مشمول به بعد إكمال سن الستين للذكر والخامسة والخمسين للأنثى، وحتى إكمال المدة الموجبة لاستحقاق راتب تقاعد الشيخوخة. وعلى المنشأة الاستمرار في شموله إذا استمر بالعمل لديها أو استخدمت خدماته بعد هذه السن، وتلتزم بدفع جميع الاشتراكات المترتبة، وتُعتمد نسب الاحتساب على أساس سن الستين للذكر والخامسة والخمسين للأنثى الواردة في الجدول رقم (6) الملحق بالقانون."),

    ("هل يجوز للمؤمن عليه إضافة المدة الناقصة لاستحقاق راتب تقاعد الشيخوخة مقابل مبلغ يدفعه؟",
     "ssl", ["65"], "eligibility", "hard",
     "نعم. للمؤمن عليه الذي أكمل سن الستين للذكر أو الخامسة والخمسين للأنثى أو استمر بعد ذلك ولم يكمل الحد الأدنى من الاشتراكات اللازمة لاستحقاق راتب تقاعد الشيخوخة وانتهت خدماته لأي سبب كان، أن يطلب إضافة المدة اللازمة لتمكينه من إكمال المدة الموجبة للاستحقاق، مقابل مبلغ يُحتسب وفقاً للجدول رقم (3) الملحق بالقانون وعلى أساس أجره الشهري الأخير الخاضع للاقتطاع المعتمد لدى المؤسسة. ويجوز له استخدام رصيد حسابه الادخاري المنصوص عليه في الفقرة (أ) من المادة 57 لتسديد المبالغ المستحقة عن هذه المدة الإضافية."),

    ("ما شروط استحقاق راتب تقاعد الوفاة الطبيعية إذا توفي المؤمن عليه أثناء خدمته؟",
     "ssl", ["66"], "eligibility", "medium",
     "يستحق راتب تقاعد الوفاة الطبيعية إذا وقعت الوفاة خلال الخدمة المشمولة بأحكام القانون، شريطة تسديد ما لا يقل عن 24 اشتراكاً فعلياً منها 6 اشتراكات متصلة. ويُحسب الراتب بنسبة 50% من متوسط الأجر الشهري الذي اتُّخذ أساساً لتسديد الاشتراكات خلال الاثني عشر اشتراكاً الأخيرة، ويُزاد بنسبة 0.5% عن كل سنة من سنوات اشتراك المؤمن عليه إذا بلغ عدد اشتراكاته 60 اشتراكاً فأكثر، وترتفع هذه النسبة إلى 1% إذا بلغت اشتراكاته 120 فأكثر."),

    ("ما شروط استحقاق راتب اعتلال العجز الطبيعي غير الناشئ عن إصابة عمل؟",
     "ssl", ["67"], "eligibility", "hard",
     "يشترط تحقق أربعة أمور مجتمعة:\n1. انتهاء خدمة المؤمن عليه.\n2. أن يتقدم بطلب تخصيص راتب الاعتلال خلال مدة لا تتجاوز ستة أشهر من تاريخ انتهاء الخدمة.\n3. أن لا تقل اشتراكاته الفعلية عن 60 اشتراكاً، منها 24 اشتراكاً متصلاً.\n4. ثبوت حالة العجز بقرار من المرجع الطبي."),
]
print(len(SCL_AR))

15


In [4]:
# --- Labor Law 1996, ENGLISH, 13 questions, worker-facing only ---
LABOR_EN = [
    # the multi-article flagship: every leave type in one answer
    ("What types of leave is a worker entitled to, and which of them are paid?",
     "labor96", ["61", "65", "66", "70"], "rights", "hard",
     "Four articles combine. Article 61 -- annual leave: 14 days on full pay per year of service, rising to 21 days once the worker completes five consecutive years with the same employer; official holidays, religious holidays and weekends are not counted against it; a worker with less than a full year gets paid leave in proportion to time worked; leave may be deferred by agreement to the immediately following year, the employer may not refuse a request to take it, and refusal for two years costs the employer a JOD 100 fine plus a cash substitute. Article 65 -- sick leave: 14 days per year on full pay on a report from the institution's approved physician, renewable for a further 14 days on full pay if the worker is hospitalised. Article 66 -- additional leaves: 14 paid days per year for an approved labour training course, or to perform the Hajj (requiring five continuous years with the employer, granted once in the service period); four months UNPAID for study at an officially recognised university, institute or college; and three days paid paternity leave. Article 70 -- maternity leave: ten weeks fully paid in total before and after childbirth, of which not less than six weeks must fall after the birth."),

    ("What notice period is required to terminate an unlimited-term employment contract?",
     "labor96", ["23"], "termination", "easy",
     "At least one month's written notice from either party. The contract remains valid throughout the notice period; the employer may excuse attendance but must still pay for it, and a worker who quits without serving the notice owes the employer for the unworked portion."),

    ("What remedy is available to a worker whose termination is found to be arbitrary?",
     "labor96", ["25"], "termination", "medium",
     "If the competent court finds, on a lawsuit filed by the worker within 60 days of termination, that the dismissal was arbitrary and in violation of this Law, it may order the employer either to reinstate the worker to their original position or to pay compensation equal to half a month's wage for each year of service, subject to a minimum of two months' wages -- in addition to the notice allowance and the other entitlements specified in Articles 32 and 33. Compensation is calculated on the last wage the worker received."),

    ("What is a worker entitled to if the employer terminates a fixed-term contract before its expiry?",
     "labor96", ["26"], "termination", "medium",
     "The worker is entitled to all rights and benefits stipulated in the contract, including the wages due for the remaining contract period -- unless the termination falls under one of the Article 28 dismissal cases."),

    ("In what circumstances is the employer prohibited from terminating a worker's service?",
     "labor96", ["27"], "termination", "medium",
     "Article 27(a) bars the employer from terminating a worker's service or sending a termination notice in three situations: a pregnant working woman from the sixth month of pregnancy or during maternity leave; a worker charged with military or reserve service, while performing it; and a worker during annual or sick leave, or during leave granted for worker learning or for Hajj, or during a leave agreed between the parties for devoting the worker to union work or for joining a recognised institute, college or university. Article 27(b) releases the employer from these protections if the worker took employment with another employer during any of those periods."),

    ("In what cases may an employer dismiss a worker without notice?",
     "labor96", ["28"], "termination", "medium",
     "Article 28 lists nine cases:\n1. Impersonating another person or identity, or providing forged certificates or documents to gain benefit or harm others.\n2. Failing to fulfil the obligations under the employment contract.\n3. Committing a mistake causing serious financial loss to the employer, provided the employer notifies the competent authority within five days of learning of it.\n4. Violating the institution's internal regulations, including work and worker safety conditions, after two written warnings.\n5. Absence without legitimate reason for more than twenty separate days in one year, or more than ten consecutive days, preceded by a written warning sent by registered mail and published once in a local daily newspaper.\n6. Disclosing work secrets.\n7. Being found guilty by conclusive judicial verdict of a felony or misdemeanour contravening honour and public morals.\n8. Being found at work drunk or under the influence of a narcotic or psychotropic substance, or committing an act violating public morals in the workplace.\n9. Assaulting the employer, the director in charge, a superior, another worker or any other person during or because of work, by beating or degradation."),

    ("What end-of-service remuneration is due to a worker who is not subject to the Social Security Law?",
     "labor96", ["32"], "benefits_calculation", "medium",
     "A worker not subject to the Social Security Law whose service ends for any reason is entitled to end-of-service remuneration at the rate of one month's wage for each year of service, calculated on the last wage received, with fractions of a year paid proportionally."),

    ("What is the maximum permitted probation period, and what are its effects?",
     "labor96", ["35"], "rights", "easy",
     "The probation period may not exceed three months, and a worker may not be employed on probation more than once by the same employer. During probation the employer may terminate without notice or end-of-service remuneration; if the worker is kept on after it, the probation period counts toward their service."),

    ("Within what period must wages be paid, and what deductions from wages are permitted?",
     "labor96", ["46", "47"], "rights", "hard",
     "Two articles combine. Article 46: wages and overtime must be paid within seven days of the date of maturity, and the employer may not deduct any part of them except in cases permitted by law; the worker's signature on a wage register or on a receipt does not waive their right to any further amount due under law, regulation or contract. Article 47 permits deduction only in seven cases:\n1. Recovering advances paid by the employer, with each repayment instalment capped at 10% of the wage.\n2. Refunding any amount paid in excess of entitlement.\n3. Social security subscriptions and instalments due, and deductions required under other laws.\n4. The worker's provident fund subscriptions.\n5. Agreed charges for employer-provided housing and other benefits or services.\n6. A debt collected in execution of a judicial verdict.\n7. Amounts imposed for violating the institution's internal regulations or the work contract, or for materials or tools destroyed through the worker's negligence or mistake."),

    ("What are the maximum working hours, and what compensation is due for overtime and for work on holidays?",
     "labor96", ["56", "59"], "rights", "hard",
     "Two articles combine. Article 56: no more than 8 hours a day or 48 hours a week, excluding meal and rest breaks, with total daily presence including breaks capped at 11 hours. Article 59: overtime is paid at no less than 125% of the regular wage per hour, and work on the weekly rest day or an official or religious holiday at no less than 150%."),

    ("Is a worker entitled to payment for annual leave not used before the end of service?",
     "labor96", ["63"], "rights", "easy",
     "Yes. If the worker's service ends for any reason before they use their annual leave, they are entitled to be paid the wage for the leave days not taken."),

    ("What is the duration of maternity leave, and what breastfeeding entitlement follows it?",
     "labor96", ["70", "71"], "rights", "hard",
     "Two articles combine. Article 70: ten weeks of fully paid maternity leave before and after childbirth in total, of which not less than six weeks must fall after the birth, and it is prohibited to employ her before that six-week period expires. Article 71: after the maternity leave under Article 70 ends, the working woman has the right, within one year from the date of birth, to take one or more periods per day on full pay to breastfeed her newborn, not exceeding a total of one hour per day."),

    ("What compensation is the employer liable to pay for death or total disability resulting from a work injury?",
     "labor96", ["90"], "benefits_calculation", "medium",
     "Where the Labor Law applies (workers not covered by the Social Security Law), death or total disability from a work injury obliges the employer to pay compensation equivalent to 1,200 days' wages, calculated on the worker's last wage. Partial disability is compensated in proportion to the assessed disability percentage."),
]
print(len(LABOR_EN))

13


In [5]:
# --- Labor Law 1996, ARABIC, 13 questions, worker-facing only ---
LABOR_AR = [
    # second multi-article flagship: the full annual-leave rulebook across four articles
    ("ما أحكام الإجازة السنوية من حيث مدتها وجواز تجزئتها ومصيرها إذا لم تُستعمل؟",
     "labor96", ["61", "62", "63", "64"], "rights", "hard",
     "أربع مواد تتكامل. المادة 61: أربعة عشر يوماً بأجر كامل عن كل سنة خدمة، وتصبح واحداً وعشرين يوماً إذا أمضى العامل خمس سنوات متصلة لدى صاحب العمل نفسه، ولا تُحسب أيام العطل الرسمية والأعياد الدينية والعطلة الأسبوعية من الإجازة السنوية، ومن لم تبلغ خدمته سنة يستحق إجازة بأجر بنسبة المدة التي عملها. المادة 62: إذا لم تُؤخذ الإجازة دفعة واحدة فلا يجوز أن يقل الجزء منها عن يومين في أي مرة. المادة 63: إذا انتهت خدمة العامل لأي سبب قبل أن يستعمل إجازته السنوية فيحق له تقاضي الأجر عن الأيام التي لم يستعملها. المادة 64: يُعتبر باطلاً كل اتفاق يقضي بتنازل العامل عن إجازته السنوية أو عن أي جزء منها."),

    ("كم يوماً من الإجازة المرضية يستحق العامل في السنة الواحدة؟",
     "labor96", ["65"], "rights", "easy",
     "أربعة عشر يوماً خلال السنة الواحدة بأجر كامل بناءً على تقرير من الطبيب المعتمد من قبل المؤسسة، ويجوز تمديدها أربعة عشر يوماً أخرى بأجر كامل إذا كان العامل نزيل أحد المستشفيات، مع اختلاف آلية اعتماد التقارير الطبية بحسب ما إذا كان عدد عمال المؤسسة يقل عن عشرين عاملاً أو يزيد عليه."),

    ("هل يستحق العامل إجازات أخرى غير الإجازة السنوية والإجازة المرضية؟",
     "labor96", ["66"], "rights", "medium",
     "نعم، تنص المادة 66 على ثلاثة أنواع. (أ) إجازة أربعة عشر يوماً في السنة مدفوعة الأجر في حالتين: الالتحاق بدورة للثقافة العمالية معتمدة من الوزارة أو الاتحاد العام لنقابات العمال بناءً على ترشيح صاحب العمل أو مدير المؤسسة بالتنسيق مع النقابة المعنية؛ أو قضاء فترة تدريب مهني معترف بها من جهة حكومية أو خاصة لدى صاحب العمل، ولا تُعطى هذه الإجازة إلا لمرة واحدة خلال مدة خدمته. (ب) إجازة أربعة أشهر دون أجر إذا التحق العامل للدراسة في جامعة أو معهد أو كلية معترف بها بصورة رسمية. (ج) إجازة أبوة ثلاثة أيام مدفوعة الأجر."),

    ("هل يحق للمرأة العاملة الحصول على إجازة للتفرغ لتربية أطفالها، وما شروطها؟",
     "labor96", ["67"], "rights", "medium",
     "نعم، للمرأة التي تعمل في مؤسسة تستخدم عشرة عمال أو أكثر الحق في إجازة دون أجر لمدة لا تزيد على سنة للتفرغ لتربية أطفالها، ويحق لها الرجوع إلى عملها بعد انتهاء الإجازة. وتفقد هذا الحق إذا عملت بأجر في أي مؤسسة أخرى خلال تلك المدة."),

    ("هل يحق للعامل الحصول على إجازة لمرافقة زوجه إذا انتقل إلى عمل خارج المحافظة؟",
     "labor96", ["68"], "rights", "easy",
     "نعم، لكل من الزوجين العاملين الحصول على إجازة لمرة واحدة دون أجر لمدة لا تزيد على سنتين لمرافقة زوجه إذا انتقل إلى عمل آخر يقع خارج المحافظة التي يعمل فيها داخل المملكة أو إلى عمل يقع خارجها."),

    ("في أي الحالات يجوز للعامل ترك العمل دون إشعار مع احتفاظه بحقوقه القانونية؟",
     "labor96", ["29"], "termination", "hard",
     "يحق للعامل ترك العمل دون إشعار صاحب العمل مع احتفاظه بحقوقه القانونية عن انتهاء الخدمة وما يترتب له من تعويضات عطل وضرر، شريطة تبليغ وزارة العمل خلال أسبوعين من تاريخ ترك العمل، وذلك في سبع حالات:\n1. استخدامه في عمل يختلف في نوعه اختلافاً بيّناً عن العمل المتفق عليه.\n2. استخدامه بصورة تدعو إلى تغيير محل إقامته الدائم ما لم ينص العقد على جواز ذلك.\n3. نقله إلى عمل في درجة أدنى من العمل المتفق عليه.\n4. تخفيض أجره.\n5. ثبوت بتقرير من مرجع طبي أن استمراره في العمل يهدد صحته.\n6. اعتداء صاحب العمل أو من يمثله عليه أثناء العمل أو بسببه بالضرب أو التحقير أو بأي شكل من أشكال الاعتداء الجنسي أو التحرش الجنسي.\n7. تخلف صاحب العمل عن تنفيذ أي حكم من أحكام القانون بعد تلقيه إشعاراً من جهة مختصة في الوزارة. وتعاقب الفقرة (ب) صاحب العمل أو مدير المؤسسة على الاعتداء أو التحرش الجنسي بغرامة لا تقل عن ألفي دينار ولا تزيد على خمسة آلاف دينار، وتُضاعف عند التكرار."),

    ("هل يلتزم صاحب العمل بمنح العامل شهادة خدمة عند انتهاء خدمته؟",
     "labor96", ["30"], "rights", "easy",
     "نعم. على صاحب العمل بناءً على طلب العامل أن يعطيه عند انتهاء خدمته شهادة خدمة تتضمن اسمه ونوع عمله وتاريخ التحاقه بالخدمة وتاريخ انتهائها، ويلزم كذلك برد ما أودعه العامل لديه من أوراق أو شهادات أو أدوات. وتتم مصادقة الشهادة من الوزارة وفق الأسس التي يصدرها الوزير."),

    ("ما الإجراءات التي يلتزم بها صاحب العمل إذا اقتضت ظروفه الاقتصادية أو الفنية تقليص حجم العمل؟",
     "labor96", ["31"], "termination", "hard",
     "على صاحب العمل تبليغ الوزير خطياً معززاً بالأسباب المبررة قبل اتخاذ أي إجراء. ويشكل الوزير لجنة من أطراف الإنتاج الثلاثة للتحقق من سلامة إجراءاته وتقديم توصياتها خلال مدة لا تتجاوز خمسة عشر يوماً من تاريخ التبليغ، ثم يصدر الوزير قراره خلال سبعة أيام من رفع التوصية إما بالموافقة على الإجراءات أو بإعادة النظر فيها. ولأي متضرر من قرار الوزير أن يطعن فيه خلال عشرة أيام من تبليغه لدى محكمة الاستئناف المختصة التي تصدر قرارها خلال مدة أقصاها شهر. ويتمتع العمال الذين أنهيت خدماتهم بحق العودة إلى العمل خلال سنة من تاريخ تركهم العمل إذا عاد العمل إلى طبيعته وأمكن استخدامهم، وللعامل الذي عُلّق عقد عمله أن يترك العمل دون إشعار مع احتفاظه بحقوقه القانونية عن انتهاء الخدمة."),

    ("إلى من تؤول حقوق العامل في حال وفاته؟",
     "labor96", ["34"], "rights", "easy",
     "تؤول إلى ورثته الشرعيين جميع حقوقه المقررة بمقتضى أحكام هذا القانون، بالإضافة إلى حقوقه في أي من صناديق الادخار أو التوفير أو التقاعد الخاصة بالمؤسسة والمنصوص عليها في المادة 33 من هذا القانون."),

    ("هل يجوز لصاحب العمل الاقتطاع من أجر العامل إذا تسبب بإتلاف أدوات أو آلات، وما الحد الأقصى للاقتطاع؟",
     "labor96", ["49"], "obligations", "medium",
     "إذا ثبت أن العامل تسبب في فقد أو إتلاف أدوات أو آلات أو منتجات يملكها أو يحوزها صاحب العمل أو كانت في عهدته، وكان ذلك ناشئاً عن خطأ العامل أو مخالفته لتعليمات صاحب العمل، فلصاحب العمل أن يقتطع من أجره قيمة الأشياء المفقودة أو المتلفة أو كلفة إصلاحها، على أن لا يزيد ما يُقتطع لهذا الغرض على أجر خمسة أيام في الشهر. ولصاحب العمل حق اللجوء إلى المحاكم النظامية المختصة للمطالبة بالتعويض."),

    ("هل يستحق العامل أجره إذا اضطر صاحب العمل إلى وقف العمل مؤقتاً لسبب لا يعزى إليه؟",
     "labor96", ["50"], "rights", "medium",
     "نعم. يستحق العامل الأجر عن مدة لا تزيد على العشرة أيام الأولى من توقف العمل خلال السنة، ويدفع له نصف الأجر عن المدة التي تزيد على ذلك، بحيث لا يزيد مجموع التعطيل الكلي المدفوع الأجر على ستين يوماً في السنة."),

    ("ما هي العطلة الأسبوعية المقررة للعامل، وهل تكون بأجر؟",
     "labor96", ["60"], "rights", "medium",
     "يوم الجمعة من كل أسبوع هو يوم العطلة الأسبوعية للعامل إلا إذا اقتضت طبيعة العمل غير ذلك، ويجوز للعامل بموافقة صاحب العمل جمع أيام عطلته الأسبوعية والحصول عليها خلال مدة لا تزيد على شهر. وتكون العطلة الأسبوعية بأجر كامل، إلا إذا كان العامل يعمل على أساس يومي أو أسبوعي فيستحق أجر يوم العطلة الأسبوعية إذا عمل ستة أيام متصلة قبل اليوم المحدد للعطلة."),

    ("ما واجبات صاحب العمل في مجال السلامة والصحة المهنية، وما الواجبات المترتبة على العامل؟",
     "labor96", ["78", "82"], "obligations", "hard",
     "مادتان تتكاملان. المادة 78: على صاحب العمل توفير الاحتياطات والتدابير اللازمة لحماية العمال من الأخطار والأمراض التي قد تنجم عن العمل وعن الآلات المستعملة فيه، وتوفير وسائل الحماية الشخصية والوقاية من أخطار العمل وأمراض المهنة كالملابس والنظارات والقفازات والأحذية وإرشاد العمال إلى طريقة استعمالها والمحافظة عليها ونظافتها، وإحاطة العامل قبل اشتغاله بمخاطر مهنته وسبل الوقاية منها. المادة 82: على العمال في أي مؤسسة التقيد بالأحكام والتعليمات والقرارات الخاصة باحتياطات الوقاية والسلامة والصحة المهنية واستعمال الأجهزة الخاصة بها والمحافظة عليها، والامتناع عن أي فعل يحول دون تنفيذ تلك الأحكام، وعن العبث بأجهزة الوقاية والسلامة أو إلحاق الضرر بها أو إتلافها، تحت طائلة العقوبات التأديبية المنصوص عليها في النظام الداخلي للمؤسسة."),
]
print(len(LABOR_AR))

13


In [6]:
# --- Professional Work Law 2019, 2 EN + 2 AR ---
# NOTE the numbering divergence: EN Article 7 == AR Article 6 (same rule, different number).
# Labelling each language with the other's number would be silently wrong -- one more reason
# this eval set has no cross-lingual questions.
PROF_EN = [
    ("Is it permitted to practise a profession in a place that has not been classified under this Law?",
     "prof19", ["5"], "scope", "easy",
     "No. It is prohibited to practise any profession in any place unless that place is classified in accordance with this Law. The business owner must also display the classification certificate prominently and keep the relevant certificates and licences for their employees."),

    ("Are professional workers classified into categories, and what follows from that classification?",
     "prof19", ["7"], "definitions", "medium",
     "Yes. Professional workers are classified into categories, and each category is assigned a name, specifications, grades and classification rules. Professional licences are issued to workers in each category in accordance with the prevailing legislation. (This is Article 7 in the English text; the same rule is Article 6 in the Arabic text.)"),
]

PROF_AR = [
    ("هل يجوز مزاولة المهنة في محل غير مصنف وفقاً لأحكام القانون؟",
     "prof19", ["5"], "scope", "easy",
     "لا يجوز مزاولة المهنة في أي محل ما لم يكن مصنفاً وفقاً لأحكام هذا القانون وحاصلاً على التراخيص المنصوص عليها في التشريعات النافذة. ويلتزم صاحب المحل بتعليق شهادة تصنيف محله في مكان بارز منه."),

    ("هل يُصنف العاملون المهنيون إلى فئات، وما الذي يترتب على هذا التصنيف؟",
     "prof19", ["6"], "definitions", "medium",
     "نعم، يُصنف العاملون المهنيون إلى فئات وتُحدد مواصفاتهم ودرجاتهم ومسمياتهم في كل فئة، وذلك بموجب إجازات مزاولة المهنة وفقاً لأحكام التشريعات النافذة. (هذه المادة 6 في النص العربي، وتقابلها المادة 7 في النص الإنجليزي.)"),
]
print(len(PROF_EN), len(PROF_AR))

2 2


## Assemble, auto-fill, and validate

`source_article_excerpt` is pulled from the knowledge base rather than typed by hand, so it cannot drift from the real text. The validation cell is the point of this notebook: it fails loudly if any ground-truth article does not exist in the index the retriever actually searches.

In [7]:
BLOCKS = [("en", SCL_EN), ("ar", SCL_AR), ("en", LABOR_EN), ("ar", LABOR_AR),
          ("en", PROF_EN), ("ar", PROF_AR)]

# Enumerations get their own line. A note listing seven conditions as one semicolon-separated
# paragraph is technically complete but unreviewable -- a human checking it against the law
# cannot see at a glance whether an item is missing. Applied here rather than baked into each
# note by hand so the rule is one place and uniform.
# Every pattern requires the marker to follow sentence-final punctuation, so a newline can
# never be inserted into the middle of a clause.
_BREAKS = [
    r"(?<=[.:؛])\s+(?=\([a-h]\)\s)",          # (a) (b) (c) (d) (h)
    r"(?<=[.:؛])\s+(?=\([أ-ي]\)\s)",  # (أ) (ب) (ج) (د)
    r"(?<=[.:؛])\s+(?=Article\s+\d+\s*(?:--|:))",
    r"(?<=[.:؛])\s+(?=المادة\s+\d+\s*:)",
    r"(?<=[.:؛])\s+(?=\d\.\s)",                # 1. 2. 3.
]

def format_note(note: str) -> str:
    for pat in _BREAKS:
        note = re.sub(pat, "\n", note)
    return note

def excerpt(law, lang, art, n=200):
    """First n characters of the real source article, whitespace-collapsed."""
    row = kb[(law, lang)].loc[art]
    return re.sub(r"\s+", " ", str(row["Content"])).strip()[:n]

rows = []
for lang, block in BLOCKS:
    for qtext, law, arts, cat, diff, note in block:
        rows.append({
            "question_id": f"q{len(rows) + 1:02d}",
            "question_text": qtext,
            "question_language": lang,
            "answer_law": law,
            "answer_article_numbers": arts,
            "answer_language": lang,          # no cross-lingual questions, by design
            "question_category": cat,
            "labeling_note": format_note(note),
            "difficulty": diff,
            "n_articles": len(arts),
            "source_article_excerpt": " || ".join(excerpt(law, lang, a) for a in arts),
        })

ev = pd.DataFrame(rows)
multi_line = (ev.labeling_note.str.count("\n") > 0).sum()
print(f"{len(ev)} questions | {multi_line} notes formatted as multi-line lists")
ev.head(3)[["question_id", "answer_law", "answer_article_numbers", "question_language"]]

60 questions | 15 notes formatted as multi-line lists


,question_id,answer_law,answer_article_numbers,question_language
0,q01,ssl,[25],en
1,q02,ssl,[26],en
2,q03,ssl,[27],en


In [8]:
chunks = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
index_keys = set(zip(chunks.law, chunks.language, chunks.article_number.astype(str)))

failures = []

# 1. every ground-truth article must exist in the index the retriever searches
for r in ev.itertuples(index=False):
    for a in r.answer_article_numbers:
        if (r.answer_law, r.answer_language, a) not in index_keys:
            failures.append(f"{r.question_id}: ({r.answer_law}, {r.answer_language}, art {a}) NOT in chunk index")

# 2. no cross-lingual questions
for r in ev.itertuples(index=False):
    if r.question_language != r.answer_language:
        failures.append(f"{r.question_id}: cross-lingual ({r.question_language} -> {r.answer_language})")

# 3. SCL questions must sit in chapters 4-8, i.e. articles 23-77 by content
for r in ev.itertuples(index=False):
    if r.answer_law == "ssl":
        for a in r.answer_article_numbers:
            if not (a.isdigit() and 23 <= int(a) <= 77):
                failures.append(f"{r.question_id}: SCL art {a} outside chapters 4-8 (23-77)")

# 4. counts must match the brief exactly
SPEC = {("ssl", "ar"): 15, ("ssl", "en"): 15, ("labor96", "ar"): 13,
        ("labor96", "en"): 13, ("prof19", "ar"): 2, ("prof19", "en"): 2}
actual = ev.groupby(["answer_law", "answer_language"]).size().to_dict()
for key, want in SPEC.items():
    got = actual.get(key, 0)
    if got != want:
        failures.append(f"count {key}: expected {want}, got {got}")

# 5. ids unique, no empty text
if ev.question_id.duplicated().any():
    failures.append("duplicate question_id")
if (ev.question_text.str.strip() == "").any() or (ev.labeling_note.str.strip() == "").any():
    failures.append("empty question_text or labeling_note")

# 6. no vague deferrals in the ground truth.
#    A note that says "as the article specifies" instead of stating the rule is useless as
#    ground truth: a model that gives the real rule cannot be credited for it, and a model
#    that punts the same way scores as correct. 13 of 60 notes failed this on the first pass
#    -- caught in review, not by the earlier checks -- so it is now enforced.
HEDGES = ["the article specifies", "the article lists", "the article sets", "sets out",
          "listed cases", "listed protected", "further cases", "the stated limits",
          "الواردة في المادة", "المفصّلة في", "المفصلة في", "التي تعددها المادة",
          "المحددة في المادة", "وفق القواعد", "الحالات الأخرى",
          "وفق ما تفصّله", "وفق ما تفصله", "كما تفصّله"]
for r in ev.itertuples(index=False):
    hit = [h for h in HEDGES if h in r.labeling_note]
    if hit:
        failures.append(f"{r.question_id}: ground truth defers instead of stating the rule -> {hit}")

print("FAILURES:", len(failures))
for f in failures:
    print(" ", f)
assert not failures, "validation failed -- do not save"
print("\nall checks passed")

FAILURES: 0

all checks passed


In [9]:
print("by law x language")
print(ev.groupby(["answer_law", "answer_language"]).size().to_string())
print("\nby difficulty")
print(ev.difficulty.value_counts().to_string())
print("\nby category")
print(ev.question_category.value_counts().to_string())
print(f"\nmulti-article questions: {(ev.n_articles > 1).sum()} of {len(ev)}")
print(ev[ev.n_articles > 1][["question_id", "answer_law", "answer_article_numbers"]].to_string(index=False))
print(f"\ncross-lingual: {(ev.question_language != ev.answer_language).sum()} (spec: 0)")
print(f"distinct articles covered: {len({(r.answer_law, r.answer_language, a) for r in ev.itertuples() for a in r.answer_article_numbers})}")

by law x language


answer_law  answer_language
labor96     ar                 13
            en                 13
prof19      ar                  2
            en                  2
ssl         ar                 15
            en                 15

by difficulty
difficulty
medium    28
easy      16
hard      16

by category
question_category
rights                  19
eligibility             10
benefits_calculation     7
termination              7
benefits                 4
obligations              4
scope                    4
procedure                3
definitions              2

multi-article questions: 8 of 60
question_id answer_law answer_article_numbers
        q10        ssl               [44, 45]
        q12        ssl               [52, 53]
        q31    labor96       [61, 65, 66, 70]
        q39    labor96               [46, 47]
        q40    labor96               [56, 59]
        q42    labor96               [70, 71]
        q44    labor96       [61, 62, 63, 64]
        q56    labor96     

In [10]:
# v1 (eval_draft.*) is left untouched -- every result from Phase 4 through 12b was measured
# against it, so it stays as the historical baseline rather than being overwritten.
JSONL = PROJECT / "eval" / "eval_v2.jsonl"
CSV = PROJECT / "eval" / "eval_v2.csv"

with open(JSONL, "w", encoding="utf-8") as f:
    for r in ev.to_dict("records"):
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
ev.to_csv(CSV, index=False, encoding="utf-8-sig")

print(f"wrote {len(ev)} questions ->")
print(" ", JSONL.name)
print(" ", CSV.name)

# round-trip check
back = [json.loads(l) for l in open(JSONL, encoding="utf-8")]
assert len(back) == 60 and back[0]["question_id"] == "q01"
assert all(isinstance(r["answer_article_numbers"], list) for r in back)
print("round-trip OK")

wrote 60 questions ->


  eval_v2.jsonl
  eval_v2.csv
round-trip OK


---

## Part 2 -- v3, written after v2 was found to be flattering every model

**Chronology note.** This half did not follow directly from the half above. In
between, `18. embeddings_evaluation_v2.ipynb` scored seven embedding models on v2
and returned a result that made no sense: five of the seven hit a perfect
recall@5 = 1.000, and the total spread across seven architecturally different
models was four questions.

That is not seven excellent models. It is a test that cannot tell anything apart.

The cause was in how v2 was written -- the questions above were written *while
reading the articles*, so they reuse the statute's own vocabulary and retrieval
gets a free ride matching a term to itself.

v3 changes **only** `question_text`. Every ground-truth field is copied and
asserted identical, so v2 and v3 scores are directly comparable and the gap
between them measures the inflation.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
V2_PATH = PROJECT / "eval" / "eval_v2.jsonl"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"

v2 = [json.loads(l) for l in open(V2_PATH, encoding="utf-8")]
chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)
print(f"loaded {len(v2)} v2 questions, {len(chunks)} articles")

loaded 60 v2 questions, 531 articles


## The rewrites

One entry per question. The rewrite describes the asker's situation in ordinary words; the ground truth it points to is untouched.

In [2]:
REWRITES = {
    # ---------- Social Security Law, English ----------
    "q01": "I was injured while at work. What does social security actually cover for me?",
    "q02": "I need an artificial limb and I have to travel for treatment. Will those costs be paid for me?",
    "q03": "A worker at my company was hurt on site today. What am I required to do straight away?",
    "q04": "I cannot work while I recover from an injury I got at work. Will I still be paid, and how much?",
    "q05": "If someone is killed or left permanently unable to work because of an accident at work, what money is paid and how is the amount worked out?",
    "q06": "Can someone lose their right to injury money because of the way they behaved?",
    "q07": "Can I also sue my employer for more than what social security pays me for my injury?",
    "q08": "I got ill from a job I left a while ago, and the symptoms only showed up now. Am I still covered?",
    "q09": "Can I take a paid job while I am receiving a disability pension for a work injury?",
    "q10": "I am pregnant and working. Do I qualify for a maternity payment, and how much would it be?",
    "q11": "I just lost my job. Do I qualify for unemployment payments?",
    "q12": "How long do unemployment payments keep coming, and does the amount drop over time?",
    "q13": "Can I draw unemployment payments at the same time as a maternity payment or a retirement pension?",
    "q14": "At what age can I stop working and draw a pension, and how many contributions do I need?",
    "q15": "Can I stop working before the normal retirement age, and how would my pension be worked out?",

    # ---------- Social Security Law, Arabic ----------
    "q16": "هل يُخصم من راتبي شيء مقابل التأمين ضد إصابات العمل، أم يدفعه صاحب العمل وحده؟",
    "q17": "أُصبت في العمل وما زلت أتعالج. إلى متى تستمر المؤسسة في تغطية علاجي؟",
    "q18": "أُصبت مرة أخرى في العمل بعد إصابة قديمة. كيف يُحسب ما أستحقه هذه المرة؟",
    "q19": "بعد أن تقرر لي راتب بسبب العجز، هل يمكن أن يُعاد فحصي طبياً مرة أخرى، ومتى؟",
    "q20": "لم أقتنع بقرار اللجنة الطبية في إصابتي. هل أستطيع الاعتراض، وكم من الوقت أمامي؟",
    "q21": "تسبب لي شخص آخر بإصابة أثناء عملي. هل تدفع لي المؤسسة أم أطالب ذلك الشخص؟",
    "q22": "أتقاضى بدل إجازة الأمومة وعُرض علي عمل خلال الإجازة. هل يتوقف البدل إن قبلت العمل؟",
    "q23": "وجدت عملاً جديداً وأنا ما زلت أتقاضى بدل التعطل. ماذا يحدث للبدل؟",
    "q24": "متى أستلم أول دفعة من بدل التعطل بعد أن تركت عملي؟",
    "q25": "هل أستطيع طلب بدل التعطل أكثر من مرة خلال حياتي العملية؟",
    "q26": "خرجت نهائياً من الضمان. ماذا يحدث للمبالغ التي اقتُطعت مني لتأمين التعطل؟",
    "q27": "بلغت الستين ولم تكتمل اشتراكاتي اللازمة للراتب التقاعدي. هل أستطيع الاستمرار في الضمان؟",
    "q28": "تنقصني مدة قصيرة حتى أستحق الراتب التقاعدي. هل أستطيع شراء هذه المدة بمبلغ أدفعه؟",
    "q29": "توفي زوجي وهو على رأس عمله. ما الشروط حتى يستحق ورثته راتباً؟",
    "q30": "أصابني عجز بسبب مرض لا علاقة له بعملي. هل أستحق راتباً من الضمان؟",

    # ---------- Labour Law 1996, English ----------
    "q31": "What time off can I take during the year, and which of it is paid?",
    "q32": "How much warning must be given before ending a job that has no fixed end date?",
    "q33": "I think I was fired for no good reason. What can I get if that is proven?",
    "q34": "My contract had a fixed end date but my employer ended it early. What am I owed?",
    "q35": "Are there situations where an employer simply is not allowed to end someone's job?",
    "q36": "When can an employer dismiss someone on the spot, with no warning at all?",
    "q37": "I am not covered by social security. What is my employer supposed to pay me when my job ends?",
    "q38": "How long can an employer keep me on trial at the start of a job, and what happens during that time?",
    "q39": "When must my wages reach me, and what is my employer allowed to take out of them?",
    "q40": "How many hours can I be asked to work, and what extra pay do I get for long hours or for working on a holiday?",
    "q41": "I did not use all my annual leave before leaving the job. Do I get paid for the days I did not take?",
    "q42": "How long can I stay off work after giving birth, and do I get time during the day for breastfeeding afterwards?",
    "q43": "If a worker is killed or left permanently unable to work by an accident at work, what is the employer required to pay?",

    # ---------- Labour Law 1996, Arabic ----------
    "q44": "كم يوم إجازة سنوية لي، وهل أستطيع تقسيمها، وماذا يحدث إن لم أستعملها؟",
    "q45": "كم يوماً أستطيع أن أتغيب بسبب المرض في السنة الواحدة؟",
    "q46": "هل هناك إجازات أخرى أستطيع أخذها غير السنوية والمرضية؟",
    "q47": "أريد التفرغ لتربية أطفالي. هل أستطيع أخذ إجازة لذلك، وبأي شروط؟",
    "q48": "نُقل زوجي للعمل في محافظة أخرى. هل أستطيع أخذ إجازة حتى أرافقه؟",
    "q49": "هل أستطيع ترك عملي فوراً دون إشعار وأحتفظ بحقوقي، وفي أي حالات؟",
    "q50": "انتهى عملي. هل يلزم صاحب العمل أن يعطيني ورقة تثبت أنني عملت لديه؟",
    "q51": "شركتنا تمر بضائقة وتريد تقليص عدد الموظفين. ما الذي يلزمها قبل أن تفعل ذلك؟",
    "q52": "إذا توفي العامل، لمن تُدفع المبالغ المستحقة له؟",
    "q53": "أتلفت آلة في العمل من غير قصد. هل يُخصم ذلك من راتبي، وكم أقصى ما يمكن خصمه؟",
    "q54": "أوقف صاحب العمل العمل مؤقتاً لسبب خارج عن إرادته. هل أتقاضى أجري في تلك الفترة؟",
    "q55": "كم يوم راحة لي في الأسبوع، وهل يكون بأجر؟",
    "q56": "ما الذي يلزم صاحب العمل حتى يحميني من مخاطر العمل، وما الذي يلزمني أنا؟",

    # ---------- Professional Work Law 2019 ----------
    "q57": "Can I work in a place that has not been given a classification yet?",
    "q58": "Am I put into a category as a professional worker, and what does that category affect?",
    "q59": "هل أستطيع ممارسة مهنتي في محل لم يُصنَّف بعد؟",
    "q60": "هل يُوضع العاملون المهنيون في فئات، وما الذي يترتب على ذلك؟",
}

assert len(REWRITES) == 60, len(REWRITES)
assert set(REWRITES) == {r["question_id"] for r in v2}, "question id mismatch"
print(f"{len(REWRITES)} rewrites defined")

60 rewrites defined


## Assemble

Ground-truth fields are copied across, then asserted identical field-by-field. A rewrite that quietly changed which article a question points to would be far worse than no rewrite at all.

In [3]:
v3 = []
for r in v2:
    new = dict(r)
    new["question_text"] = REWRITES[r["question_id"]]
    new["question_text_statute"] = r["question_text"]      # v2 wording, kept for the paired comparison
    v3.append(new)

GROUND_TRUTH_FIELDS = ["question_id", "question_language", "answer_law", "answer_article_numbers",
                       "answer_language", "question_category", "difficulty", "n_articles",
                       "labeling_note", "source_article_excerpt"]

for old, new in zip(v2, v3):
    for f in GROUND_TRUTH_FIELDS:
        assert old[f] == new[f], f"{old['question_id']}: field '{f}' changed"

# every rewrite must actually be a rewrite
unchanged = [r["question_id"] for r in v3 if r["question_text"] == r["question_text_statute"]]
assert not unchanged, f"not rewritten: {unchanged}"

# and must still point at articles that exist in the corpus
have = {(c.law, c.language, c.article_number) for c in chunks.itertuples()}
missing = [(r["question_id"], a) for r in v3 for a in r["answer_article_numbers"]
           if (r["answer_law"], r["answer_language"], str(a)) not in have]
assert not missing, f"ground truth absent from corpus: {missing}"

print(f"{len(v3)} questions rewritten, all ground truth verified unchanged")

60 questions rewritten, all ground truth verified unchanged


## Did the statute-echo actually go down?

The whole point of v3 is that questions should stop reusing the wording of the article they point to. That is measurable rather than a matter of taste: tokenise each question and its gold article, and compute what fraction of the question's content words also appear in the article.

Arabic is normalised the same way `src/retrieval.py` does it (diacritics stripped, alef/yaa/ta-marbuta folded), so the overlap figure is not distorted by orthographic variants. Stopwords are removed on both sides — matching on "the" or "من" would tell us nothing.

If v3's overlap is not clearly lower than v2's, the rewrite failed and should be redone rather than shipped.

In [4]:
AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
WORD_RE = re.compile(r"\w+", re.UNICODE)

EN_STOP = {"what", "is", "are", "the", "a", "an", "of", "to", "for", "in", "on", "at", "and", "or",
           "do", "does", "did", "can", "may", "must", "i", "my", "me", "it", "that", "this", "be",
           "am", "was", "were", "been", "have", "has", "had", "will", "would", "if", "not", "no",
           "you", "your", "they", "their", "he", "she", "his", "her", "we", "our", "so", "than",
           "how", "when", "where", "which", "who", "whom", "there", "any", "all", "some", "get",
           "got", "with", "from", "by", "as", "but", "out", "up", "about", "into", "over", "after",
           "before", "during", "while", "still", "just", "only", "also", "much", "many", "more"}
AR_STOP = {"من", "في", "على", "الى", "عن", "مع", "هل", "ما", "ماذا", "متى", "اين", "كيف", "كم",
           "الذي", "التي", "هذا", "هذه", "ذلك", "تلك", "و", "او", "ان", "انه", "اذا", "لا", "لم",
           "لن", "قد", "كان", "يكون", "هو", "هي", "انا", "لي", "له", "لها", "بعد", "قبل", "عند",
           "حتى", "ثم", "كل", "بعض", "غير", "بين", "دون", "خلال", "اثناء", "لدي", "الا", "ايضا",
           "الذين", "وهل", "وما", "به", "بها", "منه", "منها", "عليه", "عليها", "ذلكم", "هناك"}


def norm_ar(t):
    t = AR_DIACRITICS.sub("", t)
    t = re.sub(r"[آأإ]", "ا", t)
    return t.replace("ى", "ي").replace("ة", "ه")


def content_words(text):
    toks = {w.lower() for w in WORD_RE.findall(norm_ar(str(text)))}
    return {w for w in toks if w not in EN_STOP and w not in AR_STOP and len(w) > 1}


article_text = {}
for c in chunks.itertuples():
    article_text[(c.law, c.language, c.article_number)] = c.text


def echo(question, row):
    gold = set()
    for a in row["answer_article_numbers"]:
        gold |= content_words(article_text[(row["answer_law"], row["answer_language"], str(a))])
    q = content_words(question)
    return len(q & gold) / len(q) if q else 0.0


rows = []
for r in v3:
    rows.append({
        "question_id": r["question_id"],
        "lang": r["question_language"],
        "v2_echo": echo(r["question_text_statute"], r),
        "v3_echo": echo(r["question_text"], r),
    })
ec = pd.DataFrame(rows)
ec["delta"] = ec["v3_echo"] - ec["v2_echo"]

print("share of a question's content words that also appear in its own answer article")
print(f"  v2: mean {ec.v2_echo.mean():.3f}   median {ec.v2_echo.median():.3f}")
print(f"  v3: mean {ec.v3_echo.mean():.3f}   median {ec.v3_echo.median():.3f}")
print(f"  change: {ec.delta.mean():+.3f} mean\n")
print(ec.groupby("lang")[["v2_echo", "v3_echo", "delta"]].mean().round(3).to_string())
print(f"\nquestions where echo went UP (check these by hand): "
      f"{ec[ec.delta > 0].question_id.tolist() or 'none'}")
assert ec.v3_echo.mean() < ec.v2_echo.mean(), "rewrite did not reduce statute echo -- do not save"

share of a question's content words that also appear in its own answer article
  v2: mean 0.708   median 0.707
  v3: mean 0.285   median 0.273
  change: -0.424 mean

      v2_echo  v3_echo  delta
lang                         
ar      0.680    0.260 -0.420
en      0.737    0.309 -0.428

questions where echo went UP (check these by hand): ['q44', 'q55']


### The two questions the check flags

`q44` and `q55` come out with *higher* overlap in v3. Both were inspected by hand and both are fine — the metric misfires rather than the rewrite:

- **q55** — v2 used the statute's exact term **العطلة الأسبوعية**; v3 says **راحة في الأسبوع**. The statute's own word is gone, which is precisely the intended change. The ratio rose only because the question is nine words long, so a single shared word moves it a lot.
- **q44** — v3 replaced the definite statute phrase **الإجازة السنوية** with the indefinite **إجازة سنوية**. There is no lay synonym for "annual leave" that a real person would actually type; inventing one would make the question artificial and defeat the point of the rewrite.

Two known limitations of this metric, worth stating so it is not over-trusted: it does not strip Arabic clitics (`وهل` is a different token from `هل`, so some function words survive stopword removal), and on short questions the denominator is small enough that one word swings the ratio. It is a guide for spotting rewrites that failed, not a target to optimise against — driving it lower by avoiding words people genuinely use would make the eval set *less* realistic, not more.

## Save

`eval_v2.jsonl` is not touched.

In [5]:
V3_JSONL = PROJECT / "eval" / "eval_v3.jsonl"
V3_CSV = PROJECT / "eval" / "eval_v3.csv"

with open(V3_JSONL, "w", encoding="utf-8") as f:
    for r in v3:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

df = pd.DataFrame(v3)
df.to_csv(V3_CSV, index=False, encoding="utf-8-sig")

assert V2_PATH.read_text(encoding="utf-8") == "".join(
    json.dumps(r, ensure_ascii=False) + "\n" for r in v2), "eval_v2.jsonl was modified -- it must not be"

print(f"wrote {V3_JSONL.name} and {V3_CSV.name} ({len(v3)} questions)")
print(f"eval_v2.jsonl left intact: {V2_PATH.exists()}")
print("\nsample:")
for r in v3[:2] + v3[15:17] + v3[43:45]:
    print(f"  {r['question_id']} [{r['question_language']}] {r['answer_law']} art "
          f"{','.join(r['answer_article_numbers'])}")
    print(f"      v2: {r['question_text_statute']}")
    print(f"      v3: {r['question_text']}")

wrote eval_v3.jsonl and eval_v3.csv (60 questions)
eval_v2.jsonl left intact: True

sample:
  q01 [en] ssl art 25
      v2: What benefits does Work Injury Insurance provide to an injured worker?
      v3: I was injured while at work. What does social security actually cover for me?
  q02 [en] ssl art 26
      v2: Does the medical care provided under Work Injury Insurance include transport costs and prosthetic devices?
      v3: I need an artificial limb and I have to travel for treatment. Will those costs be paid for me?
  q16 [ar] ssl art 24
      v2: من يتحمل اشتراك تأمين إصابات العمل، العامل أم صاحب العمل؟
      v3: هل يُخصم من راتبي شيء مقابل التأمين ضد إصابات العمل، أم يدفعه صاحب العمل وحده؟
  q17 [ar] ssl art 28
      v2: إلى متى تبقى المؤسسة مسؤولة عن علاج المصاب بعد إصابة العمل؟
      v3: أُصبت في العمل وما زلت أتعالج. إلى متى تستمر المؤسسة في تغطية علاجي؟
  q44 [ar] labor96 art 61,62,63,64
      v2: ما أحكام الإجازة السنوية من حيث مدتها وجواز تجزئتها ومصيرها إذا لم تُستعمل؟
  

## Next

v3 is a measuring instrument, not a result. The immediate follow-on is to re-score the seven Phase 13 embedding models on it — near-free, because `eval/emb_cache/` already holds every model's corpus vectors and only the 60 new questions need embedding.

Two things that re-run will tell us:

1. **How much of Phase 13's 100% was the phrasing.** v2 and v3 have identical ground truth, so the drop is a direct measurement of the inflation, not a guess.
2. **Whether the model ranking survives.** If Cohere and Gemini stay level, Phase 13's decision stands on firmer evidence than it currently does. If the field spreads out, the eval set has its discriminating power back and the pending RRF `k`/`w` and BM25 `k1`/`b` re-tune finally has a ruler worth trusting.

Only after that is it worth asking whether a query parser earns its cost — with v3 in hand there is finally something for it to improve, and a way to prove it did.